# Telco Churn Dataset — Pandas Code Along Assignment

**Student:** Alan Munoz  
**Dataset:** `telco_churn.csv`  
**Topic:** Pandas for Data Science  

This notebook is based on the assigned Pandas tutorial and applies the same type of data science workflow to a real customer churn dataset.

The dataset used in this notebook contains **3,333 rows** and **20 columns**. The goal is to practice loading data, inspecting a DataFrame, selecting and filtering data, cleaning missing values, creating new columns, grouping data, visualizing trends, and exporting processed files.

The target column is **Churn**, which shows whether a customer left the company.


## 1. Install and Import Libraries

The first step is to import pandas. Matplotlib is also imported for simple visualizations.


In [ ]:
# Run this only if pandas is not installed:
# !pip install pandas

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Pandas version:", pd.__version__)

## 2. Load the CSV File

Place `telco_churn.csv` in the same folder as this notebook before running the cells.

The code below also includes a backup path for the ChatGPT sandbox environment.


In [ ]:
csv_path = Path("telco_churn.csv")

# Backup path used only if the notebook is running in the ChatGPT sandbox.
if not csv_path.exists():
    csv_path = Path("/mnt/data/telco_churn.csv")

df = pd.read_csv(csv_path)

print("Loaded file:", csv_path)
print("Rows and columns:", df.shape)
df.head()

## 3. Inspect the Dataset

These commands help us understand the structure of the dataset.


In [ ]:
# Show the first 5 rows
df.head()

In [ ]:
# Show the last 5 rows
df.tail()

In [ ]:
# Show all column names
df.columns

In [ ]:
# Show data types and non-null counts
df.info()

In [ ]:
# Summary statistics for numeric columns
df.describe()

## 4. Check Missing Values

Before analysis, it is important to check if the dataset has missing values.


In [ ]:
missing_values = df.isna().sum().sort_values(ascending=False)
missing_values[missing_values > 0]

This dataset has **74 total missing values**. In the cleaning step, numeric missing values will be filled with the median, and missing Churn values will be removed because Churn is the target column.


## 5. Create a DataFrame Manually

The tutorial shows that pandas can create DataFrames from dictionaries. This small example demonstrates that concept.


In [ ]:
sample_customers = {
    "Customer": ["A", "B", "C"],
    "Monthly Minutes": [250, 120, 390],
    "Churn": [False, True, False]
}

sample_df = pd.DataFrame(sample_customers)
sample_df

## 6. Select Columns and Rows

Pandas allows us to select specific columns and rows from a DataFrame.


In [ ]:
# Select one column
df["State"].head()

In [ ]:
# Select multiple columns
df[["State", "International plan", "Customer service calls", "Churn"]].head(10)

In [ ]:
# Select rows by position using iloc
df.iloc[0:5]

In [ ]:
# Select specific rows and columns by position
df.iloc[0:5, 0:5]

## 7. Filter Data

Filtering helps answer questions about specific groups of customers.


In [ ]:
# Customers with an international plan
international_customers = df[df["International plan"] == "Yes"]
international_customers.head()

In [ ]:
# Customers with 4 or more customer service calls
many_service_calls = df[df["Customer service calls"] >= 4]
many_service_calls[["State", "Customer service calls", "Churn"]].head(10)

In [ ]:
# Customers who churned
churned_customers = df[df["Churn"] == True]
churned_customers.head()

## 8. Sort Data

Sorting lets us rank customers based on usage or charges.


In [ ]:
# Sort customers by total day minutes from highest to lowest
df.sort_values(by="Total day minutes", ascending=False).head(10)

In [ ]:
# Sort customers by customer service calls
df.sort_values(by="Customer service calls", ascending=False).head(10)

## 9. Clean and Preprocess the Data

This step creates a cleaned copy of the dataset.

Cleaning choices:
- Remove rows where `Churn` is missing because it is the target column.
- Fill missing numeric values with the median.
- Fill missing text values with `"Unknown"`.
- Convert `Churn` into an integer column called `Churn_binary`.


In [ ]:
clean_df = df.copy()

# Remove rows with missing target values
clean_df = clean_df.dropna(subset=["Churn"])

# Fill missing numeric values with the median
numeric_columns = clean_df.select_dtypes(include="number").columns

for column in numeric_columns:
    clean_df[column] = clean_df[column].fillna(clean_df[column].median())

# Fill missing categorical/text values with "Unknown"
categorical_columns = clean_df.select_dtypes(include="object").columns

for column in categorical_columns:
    clean_df[column] = clean_df[column].fillna("Unknown")

# Convert Churn to 0/1
clean_df["Churn_binary"] = clean_df["Churn"].astype(bool).astype(int)

print("Original shape:", df.shape)
print("Cleaned shape:", clean_df.shape)
print("Missing values after cleaning:", clean_df.isna().sum().sum())

clean_df.head()

## 10. Add New Columns

New columns can make the dataset easier to analyze.


In [ ]:
clean_df["Total minutes"] = (
    clean_df["Total day minutes"] +
    clean_df["Total eve minutes"] +
    clean_df["Total night minutes"] +
    clean_df["Total intl minutes"]
)

clean_df["Total charges"] = (
    clean_df["Total day charge"] +
    clean_df["Total eve charge"] +
    clean_df["Total night charge"] +
    clean_df["Total intl charge"]
)

clean_df["High service calls"] = clean_df["Customer service calls"] >= 4

clean_df[["Total minutes", "Total charges", "High service calls", "Churn"]].head()

## 11. Churn Overview

This section calculates the total number of customers who churned and stayed.


In [ ]:
churn_counts = clean_df["Churn"].value_counts()
churn_rate = clean_df["Churn_binary"].mean() * 100

print(churn_counts)
print(f"Churn rate: {churn_rate:.2f}%")

## 12. Group and Aggregate Data

Grouping is useful for comparing churn rates across different customer categories.


In [ ]:
# Churn rate by international plan
international_plan_churn = clean_df.groupby("International plan")["Churn_binary"].mean().sort_values(ascending=False) * 100
international_plan_churn

In [ ]:
# Churn rate by voice mail plan
voice_mail_churn = clean_df.groupby("Voice mail plan")["Churn_binary"].mean().sort_values(ascending=False) * 100
voice_mail_churn

In [ ]:
# Churn rate by customer service calls
service_call_churn = clean_df.groupby("Customer service calls")["Churn_binary"].mean() * 100
service_call_churn

In [ ]:
# Average usage/charges grouped by churn
clean_df.groupby("Churn")[[
    "Total day minutes",
    "Total eve minutes",
    "Total night minutes",
    "Total intl minutes",
    "Total charges",
    "Customer service calls"
]].mean()

In [ ]:
# Top 10 states by churn rate
state_churn = clean_df.groupby("State")["Churn_binary"].agg(["count", "mean"])
state_churn["Churn rate %"] = state_churn["mean"] * 100
state_churn = state_churn.sort_values(by="Churn rate %", ascending=False)
state_churn.head(10)

## 13. Visualize the Data

The charts below show important churn patterns.


In [ ]:
# Churn count chart
churn_counts.plot(kind="bar", title="Customer Churn Counts")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
# Churn rate by international plan
international_plan_churn.plot(kind="bar", title="Churn Rate by International Plan")
plt.xlabel("International Plan")
plt.ylabel("Churn Rate (%)")
plt.show()

In [ ]:
# Churn rate by customer service calls
service_call_churn.plot(kind="bar", title="Churn Rate by Customer Service Calls")
plt.xlabel("Customer Service Calls")
plt.ylabel("Churn Rate (%)")
plt.show()

In [ ]:
# Distribution of total day minutes
clean_df["Total day minutes"].plot(kind="hist", bins=25, title="Distribution of Total Day Minutes")
plt.xlabel("Total Day Minutes")
plt.show()

## 14. Correlation with Churn

Correlation helps identify which numeric variables are most related to churn.


In [ ]:
correlations = clean_df.select_dtypes(include="number").corr()["Churn_binary"].sort_values(ascending=False)
correlations

## 15. Export the Cleaned Data

The tutorial explains how pandas can output data into different file formats. This notebook exports the cleaned dataset as CSV, JSON, and HTML.


In [ ]:
clean_df.to_csv("telco_churn_cleaned.csv", index=False)
clean_df.to_json("telco_churn_cleaned.json", orient="records", indent=2)
clean_df.head(50).to_html("telco_churn_preview.html", index=False)

print("Exported files:")
print("- telco_churn_cleaned.csv")
print("- telco_churn_cleaned.json")
print("- telco_churn_preview.html")

## 16. Final Reflection

In this assignment, I used pandas to analyze a Telco customer churn dataset. I loaded the CSV file into a DataFrame, inspected the dataset, checked data types and missing values, selected columns and rows, filtered customers based on service plans and customer service calls, sorted values, cleaned missing data, created new columns, grouped data, calculated churn rates, visualized patterns, checked correlations, and exported the cleaned data.

The analysis showed that churn can be studied by comparing customer usage, service calls, and plan types. Customers with more customer service calls and customers with an international plan are important groups to review because they may have a higher risk of churn. Pandas made it easier to organize and analyze the data using a clear data science workflow.
